# Adaptive Sentiment Orchestration (ASO)
### A Hybrid Framework for Real-Time Sentiment Analysis

---

**Paper:** *Adaptive Sentiment Orchestration (ASO): A Hybrid Framework for Real-Time Sentiment Analysis*

**Architecture Overview:**
```
Input Text
    │
    ▼
┌──────────────────────┐
│  Tier-1: DistilBERT  │  ──→  confidence ≥ τ  ──→  Final Output
└──────────────────────┘
    │ confidence < τ
    ▼
┌──────────────────────┐
│  Tier-2: BERT/RoBERTa│  ──────────────────────→  Final Output
└──────────────────────┘
```

**Models evaluated:**
1. Logistic Regression (TF-IDF baseline)
2. DistilBERT (Tier-1 only)
3. BERT (Tier-2 only)
4. ASO Hybrid (adaptive routing)

---
> ⚙️ **Runtime:** Enable GPU (`Runtime → Change runtime type → T4 GPU`) for best performance.

## Cell 1 — Install Dependencies

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# Run once per Colab session
# ============================================================
!pip install -q transformers datasets scikit-learn torch pandas matplotlib
!pip install -q accelerate sentencepiece
print("✅ All dependencies installed.")

## Cell 2 — Configuration & Imports

In [ ]:
# ============================================================
# CELL 2: Global Configuration & Imports
# All experiment hyper-parameters are defined here for
# reproducibility and easy ablation.
# ============================================================

import os
import sys
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

# ── Reproducibility seed ────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Dataset configuration ───────────────────────────────────
DATA_SOURCE   = "sst2"    # Options: "sst2" | "tweet_eval" | "sentiment140"
MAX_SAMPLES   = 4000      # Total samples (train + test); reduce for faster run
TEST_SIZE     = 0.20      # 20% held-out test set
CSV_PATH      = None      # Set path if DATA_SOURCE == "sentiment140"

# ── Model configuration ─────────────────────────────────────
TIER1_MODEL   = "distilbert-base-uncased-finetuned-sst-2-english"
TIER2_MODEL   = "textattack/bert-base-uncased-SST-2"
# Alternative Tier-2 (uncomment to use RoBERTa on Twitter data):
# TIER2_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# ── Router configuration ─────────────────────────────────────
ASO_THRESHOLD = 0.85      # τ: samples below this confidence go to Tier-2

# ── Inference configuration ──────────────────────────────────
INFERENCE_BATCH_SIZE = 32  # Reduce to 16 if GPU OOM errors occur

# ── Device ──────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\n{'='*50}")
print(f"  ASO Experiment Configuration")
print(f"  Data source : {DATA_SOURCE} | Max samples: {MAX_SAMPLES}")
print(f"  Tier-1      : {TIER1_MODEL}")
print(f"  Tier-2      : {TIER2_MODEL}")
print(f"  Threshold τ : {ASO_THRESHOLD}")
print(f"{'='*50}")

## Cell 3 — Load & Preprocess Dataset

In [ ]:
# ============================================================
# CELL 3: Dataset Loading & Preprocessing
# Uses setup_and_data.py (or inline if running standalone)
# ============================================================

# ── If running inside the Research paper directory: ──────────
# sys.path.insert(0, '/content/drive/MyDrive/ASO/')  # Colab Drive path
# from setup_and_data import get_data

# ── Inline implementation for standalone Colab use ───────────
import re
from datasets import load_dataset
from sklearn.model_selection import train_test_split


def clean_text(text: str) -> str:
    """Normalise text: lower-case, strip URLs, handles, special chars."""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_dataset_split(source, max_samples, test_size, seed, csv_path=None):
    if source == "sst2":
        ds = load_dataset("glue", "sst2", split="train")
        texts  = list(ds["sentence"])
        labels = list(ds["label"])
    elif source == "tweet_eval":
        ds = load_dataset("tweet_eval", "sentiment", split="train")
        texts, labels = [], []
        for t, l in zip(ds["text"], ds["label"]):
            if l == 0:   texts.append(t); labels.append(0)
            elif l == 2: texts.append(t); labels.append(1)
    elif source == "sentiment140":
        import pandas as _pd
        df = _pd.read_csv(csv_path, encoding="latin-1", header=None,
                          usecols=[0, 5], names=["polarity", "text"])
        texts  = df["text"].tolist()
        labels = (df["polarity"] == 4).astype(int).tolist()
    else:
        raise ValueError(f"Unknown source: {source}")

    if max_samples and max_samples < len(texts):
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(texts), size=max_samples, replace=False)
        texts  = [texts[i]  for i in idx]
        labels = [labels[i] for i in idx]

    texts = [clean_text(t) for t in texts]
    return train_test_split(texts, labels, test_size=test_size,
                            random_state=seed, stratify=labels)


print("Loading dataset …")
X_train, X_test, y_train, y_test = load_dataset_split(
    DATA_SOURCE, MAX_SAMPLES, TEST_SIZE, SEED, CSV_PATH
)

print(f"\nDataset: {DATA_SOURCE}")
print(f"Train samples : {len(X_train)}")
print(f"Test  samples : {len(X_test)}")
print(f"Positive ratio (test): {np.mean(y_test):.2%}")
print("\nSample texts:")
for i in range(3):
    label_str = "POS" if y_test[i] == 1 else "NEG"
    print(f"  [{label_str}] {X_test[i][:80]}…")

## Cell 4 — Baseline: Logistic Regression (TF-IDF)

In [ ]:
# ============================================================
# CELL 4: Logistic Regression Baseline
# Classic NLP benchmark using TF-IDF features.
# Provides a strong non-neural reference point.
# ============================================================

import time
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

print("Training Logistic Regression …")
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2,
        strip_accents="unicode",
        analyzer="word",
        token_pattern=r"\w{1,}",
    )),
    ("lr", LogisticRegression(
        C=1.0, max_iter=1000, solver="saga", n_jobs=-1
    )),
])

t0 = time.perf_counter()
lr_pipeline.fit(X_train, y_train)
train_time = time.perf_counter() - t0

# ── Inference with per-sample latency ─────────────────────────
latencies = []
all_preds = []
BATCH = 64
for start in range(0, len(X_test), BATCH):
    batch = X_test[start : start + BATCH]
    t0 = time.perf_counter()
    preds = lr_pipeline.predict(batch)
    elapsed = time.perf_counter() - t0
    latencies.extend([elapsed / len(batch)] * len(batch))
    all_preds.extend(preds)

# ── Metrics ────────────────────────────────────────────────────
lr_acc  = accuracy_score(y_test, all_preds)
lr_f1   = f1_score(y_test, all_preds, average="macro", zero_division=0)
lr_lat  = float(np.mean(latencies)) * 1000   # ms
lr_preds = all_preds

print(f"Training time : {train_time:.2f}s")
print(f"Accuracy      : {lr_acc:.4f}")
print(f"F1 (Macro)    : {lr_f1:.4f}")
print(f"Avg latency   : {lr_lat:.4f} ms/sample")

## Cell 5 — Tier-1: DistilBERT

In [ ]:
# ============================================================
# CELL 5: Tier-1 — DistilBERT Inference
# Pre-trained and fine-tuned on SST-2; no additional training
# required. Fast inference (~60% faster than BERT).
# ============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

print(f"Loading Tier-1 model: {TIER1_MODEL} …")
t1_tokenizer = AutoTokenizer.from_pretrained(TIER1_MODEL)
t1_model     = AutoModelForSequenceClassification.from_pretrained(TIER1_MODEL)
t1_model.to(DEVICE).eval()
print("Tier-1 model loaded.")

# Label map: DistilBERT SST-2 → {0: NEGATIVE, 1: POSITIVE}
T1_LABEL_MAP = {0: 0, 1: 1}   # Model id 0 = NEG, 1 = POS


@torch.no_grad()
def t1_predict_batch(texts):
    """Return (predictions, confidences) for a list of texts."""
    enc = t1_tokenizer(
        texts, padding=True, truncation=True,
        max_length=128, return_tensors="pt"
    ).to(DEVICE)
    logits = t1_model(**enc).logits
    probs  = F.softmax(logits, dim=-1).cpu().numpy()
    raw    = np.argmax(probs, axis=-1)
    preds  = [T1_LABEL_MAP[int(p)] for p in raw]
    confs  = [float(probs[i, raw[i]]) for i in range(len(texts))]
    return preds, confs


# ── Warm-up (discard first batch latency to avoid JIT overhead) ─
_ = t1_predict_batch(X_test[:INFERENCE_BATCH_SIZE])

# ── Full test set inference with latency tracking ───────────────
t1_preds, t1_confs, t1_latencies = [], [], []

for start in range(0, len(X_test), INFERENCE_BATCH_SIZE):
    batch = X_test[start : start + INFERENCE_BATCH_SIZE]
    t0 = time.perf_counter()
    preds, confs = t1_predict_batch(batch)
    elapsed = time.perf_counter() - t0
    lat_per_sample = elapsed / len(batch)
    t1_preds.extend(preds)
    t1_confs.extend(confs)
    t1_latencies.extend([lat_per_sample] * len(batch))

# ── Metrics ─────────────────────────────────────────────────────
t1_acc = accuracy_score(y_test, t1_preds)
t1_f1  = f1_score(y_test, t1_preds, average="macro", zero_division=0)
t1_lat = float(np.mean(t1_latencies)) * 1000

print(f"\nTier-1 (DistilBERT) Results:")
print(f"  Accuracy      : {t1_acc:.4f}")
print(f"  F1 (Macro)    : {t1_f1:.4f}")
print(f"  Avg latency   : {t1_lat:.4f} ms/sample")
print(f"  Avg confidence: {np.mean(t1_confs):.4f}")
print(f"  Samples below threshold ({ASO_THRESHOLD}): "
      f"{sum(1 for c in t1_confs if c < ASO_THRESHOLD)} / {len(t1_confs)} "
      f"({sum(1 for c in t1_confs if c < ASO_THRESHOLD)/len(t1_confs):.1%})")

## Cell 6 — Tier-2: BERT

In [ ]:
# ============================================================
# CELL 6: Tier-2 — BERT Inference
# Stronger model for ambiguous samples.
# Evaluated standalone to establish upper-bound performance.
# ============================================================

print(f"Loading Tier-2 model: {TIER2_MODEL} …")
t2_tokenizer = AutoTokenizer.from_pretrained(TIER2_MODEL)
t2_model     = AutoModelForSequenceClassification.from_pretrained(TIER2_MODEL)
t2_model.to(DEVICE).eval()
print("Tier-2 model loaded.")

# Auto-detect label map from model config
id2label = t2_model.config.id2label
print(f"Model label map: {id2label}")

def _make_label_map(id2label):
    mapping = {}
    for idx, name in id2label.items():
        nl = name.lower()
        if "neg" in nl or nl in ("label_0", "0", "false"):
            mapping[idx] = 0
        elif "pos" in nl or nl in ("label_1", "1", "true"):
            mapping[idx] = 1
        else:
            mapping[idx] = 0 if idx == min(id2label.keys()) else 1
    return mapping

T2_LABEL_MAP = _make_label_map(id2label)
print(f"Resolved label map: {T2_LABEL_MAP}")


@torch.no_grad()
def t2_predict_batch(texts):
    """Return (predictions, confidences) for a list of texts."""
    enc = t2_tokenizer(
        texts, padding=True, truncation=True,
        max_length=256, return_tensors="pt"
    ).to(DEVICE)
    logits = t2_model(**enc).logits
    probs  = F.softmax(logits, dim=-1).cpu().numpy()
    raw    = np.argmax(probs, axis=-1)
    preds  = [T2_LABEL_MAP[int(p)] for p in raw]
    confs  = [float(probs[i, raw[i]]) for i in range(len(texts))]
    return preds, confs


# ── Warm-up ─────────────────────────────────────────────────────
_ = t2_predict_batch(X_test[:INFERENCE_BATCH_SIZE])

# ── Full inference ───────────────────────────────────────────────
t2_preds_all, t2_latencies = [], []

for start in range(0, len(X_test), INFERENCE_BATCH_SIZE):
    batch = X_test[start : start + INFERENCE_BATCH_SIZE]
    t0 = time.perf_counter()
    preds, _ = t2_predict_batch(batch)
    elapsed = time.perf_counter() - t0
    t2_preds_all.extend(preds)
    t2_latencies.extend([elapsed / len(batch)] * len(batch))

t2_acc = accuracy_score(y_test, t2_preds_all)
t2_f1  = f1_score(y_test, t2_preds_all, average="macro", zero_division=0)
t2_lat = float(np.mean(t2_latencies)) * 1000

print(f"\nTier-2 (BERT) Results:")
print(f"  Accuracy      : {t2_acc:.4f}")
print(f"  F1 (Macro)    : {t2_f1:.4f}")
print(f"  Avg latency   : {t2_lat:.4f} ms/sample")

## Cell 7 — ASO: Adaptive Router

In [ ]:
# ============================================================
# CELL 7: ASO — Adaptive Routing Inference
# Core contribution: route samples by Tier-1 confidence.
# Samples below threshold τ are escalated to Tier-2.
# ============================================================

print(f"Running ASO with threshold τ = {ASO_THRESHOLD} …")

aso_preds     = []
aso_latencies = []
tier1_routed  = 0
tier2_routed  = 0

total_t0 = time.perf_counter()

for start in range(0, len(X_test), INFERENCE_BATCH_SIZE):
    batch = X_test[start : start + INFERENCE_BATCH_SIZE]

    # ── Step 1: Tier-1 inference ──────────────────────────────
    t0 = time.perf_counter()
    t1_batch_preds, t1_batch_confs = t1_predict_batch(batch)
    t1_elapsed = time.perf_counter() - t0
    t1_per_sample = t1_elapsed / len(batch)

    # ── Step 2: Partition by confidence ──────────────────────
    confident_mask   = [c >= ASO_THRESHOLD for c in t1_batch_confs]
    escalate_indices = [i for i, m in enumerate(confident_mask) if not m]
    escalate_texts   = [batch[i] for i in escalate_indices]

    final_preds = list(t1_batch_preds)  # start with Tier-1 predictions
    tier1_routed += sum(confident_mask)
    tier2_routed += len(escalate_indices)

    # ── Step 3: Escalate low-confidence samples ───────────────
    t2_elapsed = 0.0
    t2_per_sample = 0.0
    if escalate_texts:
        t0 = time.perf_counter()
        t2_batch_preds, _ = t2_predict_batch(escalate_texts)
        t2_elapsed = time.perf_counter() - t0
        t2_per_sample = t2_elapsed / len(escalate_texts)
        # Overwrite with Tier-2 predictions
        for rank, idx in enumerate(escalate_indices):
            final_preds[idx] = t2_batch_preds[rank]

    # ── Step 4: Compute per-sample latency ───────────────────
    for i in range(len(batch)):
        if confident_mask[i]:
            # Tier-1 only: proportional share of t1 batch time
            aso_latencies.append(t1_per_sample)
        else:
            # Tier-1 + Tier-2: both tiers ran
            aso_latencies.append(t1_per_sample + t2_per_sample)

    aso_preds.extend(final_preds)

total_aso_time = time.perf_counter() - total_t0

aso_acc = accuracy_score(y_test, aso_preds)
aso_f1  = f1_score(y_test, aso_preds, average="macro", zero_division=0)
aso_lat = float(np.mean(aso_latencies)) * 1000

total_samples  = len(X_test)
tier2_rate     = tier2_routed / total_samples

print(f"\nASO Results (τ = {ASO_THRESHOLD}):")
print(f"  Accuracy        : {aso_acc:.4f}")
print(f"  F1 (Macro)      : {aso_f1:.4f}")
print(f"  Avg latency     : {aso_lat:.4f} ms/sample")
print(f"  Total time      : {total_aso_time:.3f}s")
print(f"  Tier-1 answered : {tier1_routed}/{total_samples} ({1-tier2_rate:.1%})")
print(f"  Tier-2 invoked  : {tier2_routed}/{total_samples} ({tier2_rate:.1%})")

## Cell 8 — Results Table & Plots

In [ ]:
# ============================================================
# CELL 8: Final Comparison Table & Visualisations
# ============================================================

from sklearn.metrics import classification_report

# ── 8a: Summary Table ───────────────────────────────────────
results_data = [
    {"Model": "Logistic Regression (TF-IDF)",
     "Accuracy": lr_acc, "F1 Score": lr_f1,  "Avg Latency (ms)": lr_lat,
     "Notes": "Classical baseline"},
    {"Model": "DistilBERT (Tier-1 only)",
     "Accuracy": t1_acc, "F1 Score": t1_f1,  "Avg Latency (ms)": t1_lat,
     "Notes": "Fast transformer"},
    {"Model": "BERT (Tier-2 only)",
     "Accuracy": t2_acc, "F1 Score": t2_f1,  "Avg Latency (ms)": t2_lat,
     "Notes": "Strong transformer"},
    {"Model": f"ASO (τ={ASO_THRESHOLD})",
     "Accuracy": aso_acc, "F1 Score": aso_f1, "Avg Latency (ms)": aso_lat,
     "Notes": f"Hybrid; {tier2_rate:.0%} to Tier-2"},
]

df = pd.DataFrame(results_data)
df_display = df.copy()
df_display["Accuracy"]         = df["Accuracy"].map("{:.4f}".format)
df_display["F1 Score"]         = df["F1 Score"].map("{:.4f}".format)
df_display["Avg Latency (ms)"] = df["Avg Latency (ms)"].map("{:.4f}".format)

print("\n" + "="*85)
print("  ADAPTIVE SENTIMENT ORCHESTRATION — FINAL RESULTS TABLE")
print("="*85)
print(df_display.to_string(index=False))
print("="*85 + "\n")

# Render as styled HTML (for Colab notebook output)
from IPython.display import display
styled = (
    df_display.style
    .set_caption("Table 1: ASO Pipeline Model Comparison")
    .set_table_styles([{
        'selector': 'caption',
        'props': 'font-size: 14px; font-weight: bold; text-align: left;'
    }])
    .highlight_max(subset=["Accuracy", "F1 Score"], color="#d5f5d5")
    .highlight_min(subset=["Avg Latency (ms)"], color="#d5f5d5")
)
display(styled)

In [ ]:
# ============================================================
# CELL 9: Visualisation — Bar Charts
# ============================================================

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
model_names = [r["Model"] for r in results_data]
accuracies  = [r["Accuracy"]         for r in results_data]
f1_scores   = [r["F1 Score"]         for r in results_data]
latencies   = [r["Avg Latency (ms)"] for r in results_data]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    "Adaptive Sentiment Orchestration (ASO) — Model Comparison",
    fontsize=14, fontweight="bold"
)

short_names = ["LR", "DistilBERT\n(T1)", "BERT\n(T2)", f"ASO\n(τ={ASO_THRESHOLD})"]

def styled_bar(ax, values, title, ylabel, logy=False, fmt=".4f"):
    bars = ax.bar(range(len(short_names)), values,
                  color=PALETTE, width=0.55, edgecolor="white", linewidth=0.8)
    ax.set_xticks(range(len(short_names)))
    ax.set_xticklabels(short_names, ha="center", fontsize=10)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.set_ylabel(ylabel, fontsize=10)
    if logy:
        ax.set_yscale("log")
    for bar, val in zip(bars, values):
        y_off = bar.get_height() * (1.04 if not logy else 1.2)
        ax.text(bar.get_x() + bar.get_width()/2, y_off,
                f"{val:{fmt}}", ha="center", va="bottom", fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

styled_bar(axes[0], accuracies, "(a) Accuracy",         "Accuracy",     fmt=".4f")
styled_bar(axes[1], f1_scores,  "(b) F1 Score (Macro)", "F1 Score",     fmt=".4f")
styled_bar(axes[2], latencies,  "(c) Avg Latency (ms)", "Latency (ms)", logy=True, fmt=".3f")

plt.tight_layout()
plt.savefig("aso_comparison_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved: aso_comparison_plot.png")

In [ ]:
# ============================================================
# CELL 10: ASO Threshold Sweep (Ablation Study)
# Runs ASO at multiple τ values to show the accuracy-latency
# trade-off — key result for the paper.
# ============================================================

THRESHOLDS = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

sweep_rows = []
print(f"{'τ':>6} | {'Accuracy':>10} | {'F1 Macro':>10} | {'Avg Lat (ms)':>14} | {'Tier-2 (%)':>11}")
print("-" * 62)

for tau in THRESHOLDS:
    sweep_preds, sweep_lats = [], []
    t2_count = 0

    for start in range(0, len(X_test), INFERENCE_BATCH_SIZE):
        batch = X_test[start : start + INFERENCE_BATCH_SIZE]
        t1_bp, t1_bc = t1_predict_batch(batch)
        t1_time = 0.0  # already measured; use stored latencies for efficiency

        final = list(t1_bp)
        esc_idx   = [i for i, c in enumerate(t1_bc) if c < tau]
        esc_texts = [batch[i] for i in esc_idx]
        t2_count += len(esc_idx)

        if esc_texts:
            t2_bp, _ = t2_predict_batch(esc_texts)
            for rank, idx in enumerate(esc_idx):
                final[idx] = t2_bp[rank]

        sweep_preds.extend(final)

    s_acc  = accuracy_score(y_test, sweep_preds)
    s_f1   = f1_score(y_test, sweep_preds, average="macro", zero_division=0)
    t2_pct = t2_count / len(X_test) * 100
    # Approximate latency: weighted average of Tier-1 and Tier-2 latencies
    s_lat  = t1_lat * (1 - t2_pct/100) + t2_lat * (t2_pct/100)

    print(f"{tau:>6.2f} | {s_acc:>10.4f} | {s_f1:>10.4f} | {s_lat:>14.4f} | {t2_pct:>10.1f}%")
    sweep_rows.append({"Threshold": tau, "Accuracy": s_acc,
                       "F1 Macro": s_f1, "Avg Lat (ms)": s_lat,
                       "Tier-2 Rate (%)": t2_pct})

sweep_df = pd.DataFrame(sweep_rows)

# ── Plot the sweep ────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(sweep_df["Threshold"], sweep_df["Accuracy"],
         marker="o", color="#4C72B0", label="Accuracy", linewidth=2)
ax1.plot(sweep_df["Threshold"], sweep_df["F1 Macro"],
         marker="s", color="#55A868", label="F1 Macro", linewidth=2)
ax1.set_xlabel("Confidence Threshold (τ)", fontsize=11)
ax1.set_ylabel("Score", fontsize=11)
ax1.set_title("ASO Threshold Sweep — Accuracy & F1 vs. τ",
              fontsize=12, fontweight="bold")
ax1.set_ylim(0.5, 1.05)
ax1.spines["top"].set_visible(False)

ax2 = ax1.twinx()
ax2.plot(sweep_df["Threshold"], sweep_df["Tier-2 Rate (%)"],
         marker="^", color="#DD8452", linestyle="--",
         label="Tier-2 Rate (%)", linewidth=2)
ax2.set_ylabel("Tier-2 Invocation Rate (%)", color="#DD8452", fontsize=11)
ax2.tick_params(axis="y", labelcolor="#DD8452")
ax2.spines["top"].set_visible(False)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig("aso_threshold_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
print("Threshold sweep plot saved: aso_threshold_sweep.png")

In [ ]:
# ============================================================
# CELL 11: Per-Class Classification Reports
# ============================================================

target_names = ["Negative", "Positive"]
model_results = [
    ("Logistic Regression",       lr_preds),
    ("DistilBERT (Tier-1 only)",  t1_preds),
    ("BERT (Tier-2 only)",        t2_preds_all),
    (f"ASO (τ={ASO_THRESHOLD})",  aso_preds),
]

for name, preds in model_results:
    print(f"\n{'─'*55}")
    print(f"  Classification Report: {name}")
    print(f"{'─'*55}")
    print(classification_report(
        y_test, preds, target_names=target_names, zero_division=0
    ))